<a href="https://colab.research.google.com/github/ecuirty-kr/1_DataAnalysis/blob/main/p8_type2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[구글 코랩(Colab)에서 실행하기](https://colab.research.google.com/github/lovedlim/bigdata_analyst_cert_v2/blob/main/part4/ch8/p8_type2.ipynb)

In [18]:
## 문제 조건 : 통신사에서 고객에게 청구될 총 금액을 예측하시오
## 예측할 컬럼 : TotalCharges(총 청구액)
## 학습용 데이터(churn_train.csv) / 평가용 데이터(churn_test.csv)
## 제출 파일명 : "result.csv"
## 컬럼 pred : 예측된 총 청구액
## 평가지표 : MAE(Mean Absolute Error) - New!

# 1. 라이브러리 및 데이터 - 결과가 금액이니까 이번 문제는 회귀
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/churn_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/churn_test.csv")
# 데이터 샘플 확인
# print(train.head(5), "\n", test.head(5)) # 좀.. 복잡한 데이터임

# 2. 탐색적 데이터 분석 (크기 / 자료형 / 결측치 / 빈도 수 / +(기초통계량))
# print(train.shape, test.shape) # (4116, 19) (1764, 18)
# print(train.info(), test.info()) # object 15 /int 2 (TotalCharges : float in train) - 복합 데이터(범주+수치)
# print(train.isnull().sum(), test.isnull().sum()) ## 결측치 없음
# print(train['TotalCharges'].describe()) # 평균 > 중앙값 크게 치우진 데이터는 아님
# print(train['TotalCharges'].value_counts()) ## 회귀에서 확인할 자료는 아닌 듯

# 3. 데이터 전처리 (원-핫 인코딩)
target = train.pop('TotalCharges')

## ********* 'customerID' 컬럼은 삭제 해야 함 (모든 데이터데 대해 고유한 값으로 모든 값이 다 다름 : 식별자임) ************
train = train.drop('customerID', axis=1)
test = test.drop('customerID', axis=1)  ## train 데이터에서 제거한 컬럼은 test 데이터에도 동일하게 제거해야 함

# train = pd.get_dummies(train)
# test = pd.get_dummies(test)
# print(train.shape, test.shape) # (4116, 4159) (1764, 1807) 확인 -> 원-핫 인코딩 수행 시간이 지연되어 레이블 인코딩 수행

# 3-1. 레이블 인코딩
from sklearn.preprocessing import LabelEncoder
# le = LabelEncoder() ## 레이블 인코더는 반복문 수행시마다 생성돼야 함
cols = train.select_dtypes(include='object').columns # 반복문 수행 조건에 들어갈 컬럼 명
for col in cols:
  le = LabelEncoder()
  train[col] = le.fit_transform(train[col])
  test[col] = le.transform(test[col])

# 4. 검증 데이터 분리
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 5. 머신러닝 학습 및 평가 (RF)
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state=0)
rf.fit(X_train, y_train)
pred = rf.predict(X_val)

## 평가 지표 : MAE
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(y_val, pred)
print("RF mae : ", mae) # 결과 : RF mae :  951.0960435538027

# 5-1. LightGBM
import lightgbm as lgb
lg = lgb.LGBMRegressor(random_state=0, verbose=-1)
lg.fit(X_train, y_train)
pred = lg.predict(X_val)
# 평가
mae_lg = mean_absolute_error(y_val, pred)
print("LG MAE : ", mae_lg) # 결과 : LG MAE :  952.7925407798712

# 평가지표 RF < LG 이므로 적은 RF가 성능이 더 우수 (대충, 분류 = 큰 값 / 회귀 = 작은 값 으로 이해)

# 6. 예측 수행 및 결과 파일 생성
pred = rf.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv("result.csv", index=False)

## 검증
print(pred.shape) # (1764,)  : 첫 test 데이터와 크기 동일한지 확인
file = pd.read_csv("result.csv")
file  ## 확인 완료

RF mae :  951.0960435538027
LG MAE :  952.7925407798712
(1764,)


,pred
0,3707.6212
1,923.7132
2,4057.4078
3,952.6143
4,1322.1638
...,...
1759,304.8777
1760,4085.7944
1761,4509.4186
1762,988.5564


In [ ]:
# 문제정의
# 평가: MAE
# target: TotalCharges
# 최종파일: result.csv(컬럼 1개 pred)

# 라이브러리 및 데이터 불러오기
import pandas as pd
# train = pd.read_csv("churn_train.csv")
# test = pd.read_csv("churn_test.csv")
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/churn_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bigdata_analyst_cert/main/part4/ch8/churn_test.csv")

# 탐색적 데이터 분석(EDA)
print("===== 데이터 크기 =====")
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

print("\n ===== 데이터 정보(자료형) =====")
print(train.info())

print("\n ===== train 결측치 수 =====")
print(train.isnull().sum().sum())

print("\n ===== test 결측치 수 =====")
print(test.isnull().sum().sum())

print("\n ===== customerID unique 수 =====")
print(train['customerID'].nunique())

print("\n ===== target 기술 통계 =====")
print(train['TotalCharges'].describe())

===== 데이터 크기 =====
Train Shape: (4116, 19)
Test Shape: (1764, 18)

 ===== 데이터 정보(자료형) =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4116 entries, 0 to 4115
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        4116 non-null   object 
 1   gender            4116 non-null   object 
 2   SeniorCitizen     4116 non-null   int64  
 3   Partner           4116 non-null   object 
 4   Dependents        4116 non-null   object 
 5   tenure            4116 non-null   int64  
 6   PhoneService      4116 non-null   object 
 7   MultipleLines     4116 non-null   object 
 8   InternetService   4116 non-null   object 
 9   OnlineSecurity    4116 non-null   object 
 10  OnlineBackup      4116 non-null   object 
 11  DeviceProtection  4116 non-null   object 
 12  TechSupport       4116 non-null   object 
 13  StreamingTV       4116 non-null   object 
 14  StreamingMovies   4116 non-null   object 
 1

In [ ]:
# 데이터 전처리
train = train.drop('customerID', axis=1)
test = test.drop(['customerID'], axis=1)
target = train.pop('TotalCharges')

# 레이블 인코딩
from sklearn.preprocessing import LabelEncoder
cols = train.select_dtypes(include='O').columns

for col in cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

# 검증데이터 분리
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 랜덤포레스트
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)

# MAE
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(y_val, pred))

# LightGBM
import lightgbm as lgb
lg = lgb.LGBMRegressor(random_state=0, verbose=-1)
lg.fit(X_tr, y_tr)
pred = lg.predict(X_val)
print(mean_absolute_error(y_val, pred))

# 최종 제출 파일
pred = rf.predict(test)
result = pd.DataFrame({
    'pred':pred
})
result.to_csv("result.csv", index=False)

951.0960435538027
952.7925407798712


In [ ]:
# 1. pred 행의 수
print(pred.shape)

# 2. 생성한 csv 확인
print(pd.read_csv("result.csv").head())

(1764,)
        pred
0  3707.6212
1   923.7132
2  4057.4078
3   952.6143
4  1322.1638
